# Notebook 01 — Data preparation

Walk through how the raw documentation in `data/raw/` is converted into the
Qdrant + BM25 indexes that the RAG pipeline uses at query time.

Steps implemented by `src/ingest.py` and `src/index.py`:

1. **Collect** — `iter_markdown_files()` walks every `.md` file under `data/raw/`,
   skipping empty placeholders and Hugo navigation stubs.
2. **Clean** — `clean_markdown()` strips front-matter, shortcodes, HTML,
   image syntax, and collapses blank lines.
3. **Chunk** — `chunk_text()` produces overlapping 500-token chunks with
   80-token overlap using the `cl100k_base` tokenizer.
4. **Embed** — `embed_texts()` calls `text-embedding-3-small` in 96-row
   batches.
5. **Upsert** — content-hashed point IDs keep the operation idempotent
   (re-running the pipeline merges, never duplicates).
6. **Persist** — `data/chunks.jsonl` for the BM25 index, `data/bm25.pkl`
   for the sparse ranker.

Run end-to-end with:

```bash
PYTHONPATH=src python src/ingest.py run
PYTHONPATH=src python src/index.py
```


In [ ]:
from pathlib import Path
import json
REPO = Path.cwd()
raw_manifest = REPO / 'data' / 'raw_manifest.jsonl'
chunks = REPO / 'data' / 'chunks.jsonl'
print('manifest exists:', raw_manifest.exists())
print('chunks exist:', chunks.exists())
if raw_manifest.exists():
    n = sum(1 for _ in raw_manifest.open())
    print('documents in manifest:', n)
if chunks.exists():
    n = sum(1 for _ in chunks.open())
    print('chunks in JSONL:', n)
